# IntelliDoc AI — Notebook 6: LayoutLMv3 Fine-Tuning on CORD

**Purpose:** Fine-tune `microsoft/layoutlmv3-base` on the **CORD (Consolidated Receipt Dataset)** training set and evaluate on the test set. Compare results against the Notebook 5 baseline.

**Research Objective:**
1. Demonstrates fine-tuning gains on receipt line-items (`MENU`, `SUBTOTAL`, `TOTAL`).
2. Produces an empirical checkpoint for receipt line-item extraction in IntelliDoc AI.

---
> **Prerequisites:** Run Notebook 4 (Validation) and Notebook 5 (Baseline) first.
> **Runtime:** ~20–35 minutes on a free Google Colab T4 GPU.

## Step 1: Mount Drive, Set Paths, Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

CORD_ROOT = '/content/drive/MyDrive/Intellidoc_AI/datasets_public/cord'
if not os.path.exists(CORD_ROOT):
    CORD_ROOT = '/content/drive/MyDrive/cord'

TRAIN_DIR = os.path.join(CORD_ROOT, 'train')
DEV_DIR   = os.path.join(CORD_ROOT, 'dev')
TEST_DIR  = os.path.join(CORD_ROOT, 'test')

CHECKPOINT_DIR = '/content/drive/MyDrive/intellidoc_checkpoints/layoutlmv3_cord'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for path in [TRAIN_DIR, DEV_DIR, TEST_DIR]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing split: {path}')

print('✅ Paths verified. Model checkpoints will be saved to:', CHECKPOINT_DIR)

!pip install -q transformers datasets seqeval Pillow evaluate accelerate
print('✅ Dependencies installed!')

## Step 2: CORD BIO Scheme & Data Preprocessing

In [ ]:
ENTITIES = ['MENU', 'SUBTOTAL', 'TOTAL', 'VOID']
LABELS = ['O']
for ent in ENTITIES:
    LABELS.append(f'B-{ent}')
    LABELS.append(f'I-{ent}')

ID2LABEL = {i: label for i, label in enumerate(LABELS)}
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

print(f'Configured {NUM_LABELS} BIO Labels: {LABELS}')

In [ ]:
import json
import glob
from PIL import Image

def normalize_bbox(quad, width, height):
    xs = [quad.get('x1', 0), quad.get('x2', 0), quad.get('x3', 0), quad.get('x4', 0)]
    ys = [quad.get('y1', 0), quad.get('y2', 0), quad.get('y3', 0), quad.get('y4', 0)]
    return [
        max(0, min(1000, int(1000 * (min(xs) / width)))),
        max(0, min(1000, int(1000 * (min(ys) / height)))),
        max(0, min(1000, int(1000 * (max(xs) / width)))),
        max(0, min(1000, int(1000 * (max(ys) / height))))
    ]

def map_category_to_entity(cat):
    if 'menu' in cat:
        return 'MENU'
    elif 'sub_total' in cat:
        return 'SUBTOTAL'
    elif 'total' in cat:
        return 'TOTAL'
    elif 'void' in cat:
        return 'VOID'
    return None

def load_cord_samples(split_dir, limit=None):
    json_files = sorted(glob.glob(os.path.join(split_dir, 'json', '*.json')))
    if limit:
        json_files = json_files[:limit]
        
    samples = []
    for j_path in json_files:
        stem = os.path.basename(j_path).replace('.json', '')
        img_path = os.path.join(split_dir, 'image', stem + '.png')
        if not os.path.exists(img_path):
            img_path = os.path.join(split_dir, 'image', stem + '.jpg')
        if not os.path.exists(img_path):
            continue
        
        with open(j_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        img = Image.open(img_path).convert('RGB')
        width, height = img.size
        words, bboxes, labels = [], [], []
        
        for line in data.get('valid_line', []):
            cat = line.get('category', '')
            entity = map_category_to_entity(cat)
            line_words = line.get('words', [])
            for i, w in enumerate(line_words):
                t = w.get('text', '').strip()
                quad = w.get('quad', {})
                if not t or not quad:
                    continue
                box = normalize_bbox(quad, width, height)
                tag = 'O'
                if entity:
                    prefix = 'B' if i == 0 else 'I'
                    tag = f'{prefix}-{entity}'
                
                words.append(t)
                bboxes.append(box)
                labels.append(LABEL2ID.get(tag, 0))
        
        if words:
            samples.append({
                'image': img,
                'words': words,
                'bboxes': bboxes,
                'labels': labels
            })
    return samples

print('Loading CORD training and dev samples...')
train_samples = load_cord_samples(TRAIN_DIR)
dev_samples   = load_cord_samples(DEV_DIR)
test_samples  = load_cord_samples(TEST_DIR)

print(f'Train Samples : {len(train_samples)}')
print(f'Dev Samples   : {len(dev_samples)}')
print(f'Test Samples  : {len(test_samples)}')

## Step 3: PyTorch Dataset Wrapper & Processor

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import LayoutLMv3Processor

MODEL_NAME = 'microsoft/layoutlmv3-base'
processor  = LayoutLMv3Processor.from_pretrained(MODEL_NAME, apply_ocr=False)

class CORDDataset(Dataset):
    def __init__(self, samples, processor, max_length=512):
        self.samples    = samples
        self.processor  = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        encoding = self.processor(
            sample['image'],
            sample['words'],
            boxes=sample['bboxes'],
            word_labels=sample['labels'],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {k: v.squeeze(0) for k, v in encoding.items()}

train_dataset = CORDDataset(train_samples, processor)
dev_dataset   = CORDDataset(dev_samples, processor)
test_dataset  = CORDDataset(test_samples, processor)

print('PyTorch Datasets ready for training.')

## Step 4: Model Configuration & Training

In [ ]:
from transformers import LayoutLMv3ForTokenClassification, TrainingArguments, Trainer
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
import numpy as np

model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    true_predictions = [
        [ID2LABEL[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [ID2LABEL[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    return {
        'precision': precision_score(true_labels, true_predictions, zero_division=0),
        'recall': recall_score(true_labels, true_predictions, zero_division=0),
        'f1': f1_score(true_labels, true_predictions, zero_division=0),
    }

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=8,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

print('Starting LayoutLMv3 fine-tuning on CORD receipts...')
trainer.train()

## Step 5: Test Set Evaluation & Final Checkpoint Export

In [ ]:
print('=== Evaluating Best Checkpoint on CORD Test Set ===')
test_results = trainer.evaluate(test_dataset)
for k, v in test_results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

# Save final model weights
final_save_path = os.path.join(CHECKPOINT_DIR, 'final_model')
model.save_pretrained(final_save_path)
processor.save_pretrained(final_save_path)

print('\n' + '=' * 60)
print(f'✅ Fine-Tuned Model Saved to: {final_save_path}')
print('=' * 60)